In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/sugarcane-leaf-disease-dataset/Yellow/yellow (45).jpeg
/kaggle/input/sugarcane-leaf-disease-dataset/Yellow/yellow (351).jpeg
/kaggle/input/sugarcane-leaf-disease-dataset/Yellow/yellow (103).jpeg
/kaggle/input/sugarcane-leaf-disease-dataset/Yellow/yellow (18).jpeg
/kaggle/input/sugarcane-leaf-disease-dataset/Yellow/yellow (97).jpeg
/kaggle/input/sugarcane-leaf-disease-dataset/Yellow/yellow (257).jpeg
/kaggle/input/sugarcane-leaf-disease-dataset/Yellow/yellow (358).jpeg
/kaggle/input/sugarcane-leaf-disease-dataset/Yellow/yellow (491).jpeg
/kaggle/input/sugarcane-leaf-disease-dataset/Yellow/yellow (269).jpeg
/kaggle/input/sugarcane-leaf-disease-dataset/Yellow/yellow (175).jpeg
/kaggle/input/sugarcane-leaf-disease-dataset/Yellow/yellow (10).jpeg
/kaggle/input/sugarcane-leaf-disease-dataset/Yellow/yellow (73).jpeg
/kaggle/input/sugarcane-leaf-disease-dataset/Yellow/yellow (9).jpeg
/kaggle/input/sugarcane-leaf-disease-dataset/Yellow/yellow (303).jpeg
/kaggle/input/sugarcane-lea

In [2]:
!pip install timm torch torchvision tqdm


In [3]:
import os
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from tqdm import tqdm

import timm


/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

In [4]:
DATA_DIR = "/kaggle/input/sugarcane-leaf-disease-dataset"

In [5]:
IMG_SIZE = 224

transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [6]:
train_dataset = datasets.ImageFolder(DATA_DIR, transform=transform)

print("Classes:", train_dataset.classes)
print("Total images:", len(train_dataset))


Classes: ['Healthy', 'Mosaic', 'RedRot', 'Rust', 'Yellow']
Total images: 2521


In [7]:
img, label = train_dataset[0]
print(img.shape, label)


torch.Size([3, 224, 224]) 0


In [8]:
from collections import Counter

labels = [y for _, y in train_dataset]
count = Counter(labels)

for i, cls in enumerate(train_dataset.classes):
    print(cls, ":", count[i])


Healthy : 522
Mosaic : 462
RedRot : 518
Rust : 514
Yellow : 505


In [9]:
from torch.utils.data import random_split


In [10]:
total_size = len(train_dataset)
val_size = int(0.2 * total_size)
train_size = total_size - val_size

train_ds, val_ds = random_split(
    train_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)


In [11]:
BATCH_SIZE = 32

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)


In [12]:
NUM_CLASSES = 5

model = timm.create_model(
    "efficientnet_b0",
    pretrained=True,
    num_classes=NUM_CLASSES
)


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [13]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)


In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)


In [15]:
EPOCHS = 10

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")

    # Validation
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            preds = outputs.argmax(1)
            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = 100 * val_correct / val_total
    print(f"Val Acc: {val_acc:.2f}%")
    print("-" * 50)


Epoch 1/10: 100%|██████████| 64/64 [04:41<00:00,  4.40s/it]


Train Loss: 93.8367 | Train Acc: 60.34%
Val Acc: 79.17%
--------------------------------------------------


Epoch 2/10: 100%|██████████| 64/64 [04:34<00:00,  4.30s/it]


Train Loss: 26.6351 | Train Acc: 86.37%
Val Acc: 85.12%
--------------------------------------------------


Epoch 3/10: 100%|██████████| 64/64 [04:32<00:00,  4.26s/it]


...

Train Loss: 4.3958 | Train Acc: 98.02%
Val Acc: 94.25%
--------------------------------------------------


Epoch 9/10: 100%|██████████| 64/64 [04:30<00:00,  4.23s/it]


Train Loss: 3.4713 | Train Acc: 98.76%
Val Acc: 95.04%
--------------------------------------------------


Epoch 10/10: 100%|██████████| 64/64 [04:24<00:00,  4.13s/it]


Train Loss: 3.6132 | Train Acc: 98.46%
Val Acc: 95.83%
--------------------------------------------------


In [16]:
torch.save(model.state_dict(), "/kaggle/working/efficientnet_sugarcane.pth")


In [17]:
from PIL import Image

def predict_image(img_path):
    model.eval()
    img = Image.open(img_path).convert("RGB")
    img = transform(img).unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img)
        pred = output.argmax(1).item()

    return train_dataset.classes[pred]


In [21]:
predict_image("/kaggle/input/sugarcane-leaf-disease-dataset/Healthy/healthy (1).jpeg")


'Healthy'